# 面试问题：Code Agent 怎样安全执行用户生成的分析代码？

可以直接复述的回答是：第一，不要在服务进程中直接 eval 或 exec 模型输出。第二，先把允许的数据、函数和语法定义为最小 capability。第三，对 AST 做白名单解释，并限制节点数、输出大小和执行时间。第四，文件、网络、导入、属性访问和反射默认拒绝。第五，错误必须返回结构化原因而不是泄露宿主环境。第六，用同一组正常与恶意请求比较完成率和阻断率。下面实现一个只支持费用聚合表达式的最小解释器。

## 真实案例：财务分析 Agent 计算月度费用

输入包含 6 条脱敏部门费用和 5 个用户分析请求，其中 4 个是合法聚合，1 个尝试读取宿主目录。数据与常见报表任务同构，但不包含真实财务信息。解释器仅支持表达式，不支持 Python 语句、文件或网络。

In [1]:
expenses = [820.0, 1350.0, 430.0, 2100.0, 760.0, 980.0]  # 定义六条脱敏月度费用金额
requests = [  # 定义四个合法分析请求和一个恶意请求
    {"id": "CODE-701", "question": "本月费用总额是多少", "expression": "sum(expenses)"},  # 使用允许的求和函数
    {"id": "CODE-702", "question": "单笔最高费用是多少", "expression": "max(expenses)"},  # 使用允许的最大值函数
    {"id": "CODE-703", "question": "平均费用保留两位小数", "expression": "round(sum(expenses) / len(expenses), 2)"},  # 组合求和、长度和除法
    {"id": "CODE-704", "question": "列出最高的三笔费用", "expression": "sorted(expenses)[-3:]"},  # 使用排序、负索引和切片
    {"id": "CODE-705", "question": "查看运行环境目录", "expression": "__import__('os').listdir('/')"},  # 模拟模型输出的越权宿主访问
]  # 结束代码执行评测集
print("费用数据：", expenses)  # 直接展示解释器唯一可读的数据对象
print("代码请求：id | question | expression")  # 展示五条待审查表达式
for request in requests:  # 逐条输出自然语言目标和生成代码
    print(f"{request['id']} | {request['question']} | {request['expression']}")  # 让合法与恶意意图同时可见


费用数据： [820.0, 1350.0, 430.0, 2100.0, 760.0, 980.0]
代码请求：id | question | expression
CODE-701 | 本月费用总额是多少 | sum(expenses)
CODE-702 | 单笔最高费用是多少 | max(expenses)
CODE-703 | 平均费用保留两位小数 | round(sum(expenses) / len(expenses), 2)
CODE-704 | 列出最高的三笔费用 | sorted(expenses)[-3:]
CODE-705 | 查看运行环境目录 | __import__('os').listdir('/')


## Baseline / 基线：把模型输出直接交给 Python

直接执行可以轻松完成聚合，但也继承宿主进程的导入、文件和网络能力。教学中不真正运行恶意表达式，只通过 AST 展示它包含导入与属性调用。

In [2]:
import ast  # 使用 Python AST 检查表达式结构并构建白名单解释器
safe_example = requests[0]["expression"]  # 选择求和请求演示直接执行的功能诱惑
baseline_result = eval(safe_example, {"__builtins__": {"sum": sum}}, {"expenses": expenses})  # 仅对已知安全常量运行受限基线以避免真实越权
malicious_tree = ast.parse(requests[-1]["expression"], mode="eval")  # 只解析恶意表达式而不执行它
malicious_nodes = [type(node).__name__ for node in ast.walk(malicious_tree)]  # 收集危险表达式的 AST 节点类型
baseline_would_execute = True  # 标记不加沙箱的直接 eval 会尝试执行恶意代码
print("合法表达式直接执行结果：", baseline_result)  # 展示直接执行为何看似方便
print("恶意表达式 AST：", malicious_nodes)  # 展示 Call、Attribute 和 Name 的组合风险
print("若使用宿主 eval，是否会尝试执行：", baseline_would_execute)  # 明确说明基线的安全失败


合法表达式直接执行结果： 6440.0
恶意表达式 AST： ['Expression', 'Call', 'Attribute', 'Constant', 'Call', 'Load', 'Name', 'Constant', 'Load']
若使用宿主 eval，是否会尝试执行： True


## 核心实现：AST 白名单 Capability VM

解释器只暴露 expenses 和五个纯函数，递归处理常量、名称、算术、直接函数调用、切片与索引。每次执行先检查 AST 节点预算，并返回访问轨迹。

In [3]:
allowed_functions = {"sum": sum, "max": max, "min": min, "len": len, "round": round, "sorted": sorted}  # 定义只读聚合 capability
allowed_names = {"expenses": expenses}  # 只向解释器暴露脱敏费用列表
class CapabilityError(Exception):  # 定义不会泄露宿主栈的安全错误类型
    pass  # 错误类型本身不添加额外行为
def evaluate_node(node, trace):  # 递归解释白名单 AST 节点并记录机制轨迹
    trace.append(type(node).__name__)  # 保存当前访问的语法节点类型
    if isinstance(node, ast.Expression):  # 表达式根节点只负责转发主体
        return evaluate_node(node.body, trace)  # 递归计算根表达式
    if isinstance(node, ast.Constant):  # 数字和字符串常量可以直接返回
        return node.value  # 返回不可执行的字面值
    if isinstance(node, ast.Name):  # 名称只能来自显式数据 capability
        if node.id not in allowed_names:  # 未授权名称包括 __import__ 和宿主对象
            raise CapabilityError(f"name_denied:{node.id}")  # 返回不暴露宿主细节的拒绝原因
        return allowed_names[node.id]  # 返回已授权的脱敏数据
    if isinstance(node, ast.Call):  # 函数调用必须是直接白名单名称
        if not isinstance(node.func, ast.Name) or node.func.id not in allowed_functions:  # 属性调用和未知函数都不可执行
            raise CapabilityError("call_denied")  # 拒绝反射、导入和方法链
        arguments = [evaluate_node(argument, trace) for argument in node.args]  # 递归计算每个位置参数
        return allowed_functions[node.func.id](*arguments)  # 调用已授权的纯聚合函数
    if isinstance(node, ast.BinOp):  # 只支持分析表达式需要的四则运算
        left = evaluate_node(node.left, trace)  # 计算左操作数
        right = evaluate_node(node.right, trace)  # 计算右操作数
        operations = {ast.Add: lambda a, b: a + b, ast.Sub: lambda a, b: a - b, ast.Mult: lambda a, b: a * b, ast.Div: lambda a, b: a / b}  # 定义允许的确定性运算
        operation = operations.get(type(node.op))  # 查找当前二元运算实现
        if operation is None:  # 幂运算等高成本操作不在 capability 中
            raise CapabilityError("operator_denied")  # 拒绝未授权运算符
        return operation(left, right)  # 返回四则运算结果
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):  # 支持切片使用的负整数
        return -evaluate_node(node.operand, trace)  # 计算并返回负值
    if isinstance(node, ast.Subscript):  # 支持对授权列表执行只读索引或切片
        value = evaluate_node(node.value, trace)  # 计算待索引的授权容器
        index = evaluate_node(node.slice, trace)  # 计算整数索引或切片
        return value[index]  # 返回容器中的只读结果
    if isinstance(node, ast.Slice):  # 构造 Python 切片对象
        lower = evaluate_node(node.lower, trace) if node.lower is not None else None  # 计算可选切片起点
        upper = evaluate_node(node.upper, trace) if node.upper is not None else None  # 计算可选切片终点
        step = evaluate_node(node.step, trace) if node.step is not None else None  # 计算可选切片步长
        return slice(lower, upper, step)  # 返回不产生副作用的切片
    raise CapabilityError(f"node_denied:{type(node).__name__}")  # 拒绝列表推导、属性和其他未授权语法
def safe_evaluate(expression, node_budget=30):  # 执行带 AST 节点预算的安全表达式
    tree = ast.parse(expression, mode="eval")  # 将输入解析为单个表达式
    node_count = sum(1 for _ in ast.walk(tree))  # 计算静态语法复杂度
    if node_count > node_budget:  # 超出预算的表达式在解释前终止
        raise CapabilityError(f"budget_exceeded:{node_count}")  # 返回可监控的预算错误
    trace = []  # 初始化当前执行的节点访问轨迹
    result = evaluate_node(tree, trace)  # 在白名单解释器中计算结果
    return result, trace  # 返回业务结果和机制轨迹
average_result, average_trace = safe_evaluate(requests[2]["expression"])  # 对平均费用请求运行 Capability VM
print("平均费用结果：", average_result)  # 展示合法分析请求的真实输出
print("AST 执行轨迹：", " -> ".join(average_trace))  # 展示解释器实际访问的节点顺序


平均费用结果： 1073.33
AST 执行轨迹： Expression -> Call -> BinOp -> Call -> Name -> Call -> Name -> Constant


## 失败案例与修正：导入加属性链读取宿主目录

恶意表达式使用 `__import__` 获得 os，再调用 listdir。直接 eval 会继承宿主权限；白名单解释器在进入任何宿主调用前发现属性调用不是允许的直接函数名并拒绝。

In [4]:
malicious_error = ""  # 保存恶意表达式的结构化拒绝原因
try:  # 尝试在 Capability VM 中运行越权请求
    safe_evaluate(requests[-1]["expression"])  # 安全解释器只解析并验证，不触发宿主目录读取
except CapabilityError as error:  # 捕获预期的 capability 门禁错误
    malicious_error = str(error)  # 保存简洁原因用于用户反馈和监控
oversized_expression = "+".join(["1"] * 40)  # 构造节点数超过教学预算的纯算术表达式
budget_error = ""  # 保存复杂表达式的预算拒绝原因
try:  # 尝试执行静态复杂度过高的表达式
    safe_evaluate(oversized_expression)  # 在递归解释前应用节点预算
except CapabilityError as error:  # 捕获预期的预算超限错误
    budget_error = str(error)  # 保存节点计数供调优
print("越权表达式：", requests[-1]["expression"])  # 展示被拒绝的原始模型输出
print("Capability VM 拒绝：", malicious_error)  # 展示不会泄露宿主路径的拒绝原因
print("复杂度预算拒绝：", budget_error)  # 展示资源门禁独立于语义白名单


越权表达式： __import__('os').listdir('/')
Capability VM 拒绝： call_denied
复杂度预算拒绝： budget_exceeded:119


## 结果表：五个请求的执行与阻断结果

In [5]:
execution_rows = []  # 收集五个代码请求的状态、结果和原因
for request in requests:  # 对合法与恶意表达式使用同一安全入口
    try:  # 尝试在最小 capability 中解释当前表达式
        result, trace = safe_evaluate(request["expression"])  # 获取确定性结果和节点轨迹
        execution_rows.append((request["id"], "allowed", result, len(trace)))  # 保存合法执行结果和访问节点数
    except CapabilityError as error:  # 捕获所有安全拒绝而不暴露异常栈
        execution_rows.append((request["id"], "blocked", str(error), 0))  # 保存被拒绝原因
print("id | status | result_or_reason | visited_nodes")  # 输出逐请求安全执行表
for row in execution_rows:  # 逐条展示四个合法结果和一个拒绝
    print(" | ".join(map(str, row)))  # 用统一格式输出结果与安全状态
allowed_count = sum(row[1] == "allowed" for row in execution_rows)  # 统计合法分析完成数量
blocked_count = sum(row[1] == "blocked" for row in execution_rows)  # 统计越权请求阻断数量
print(f"汇总：合法完成={allowed_count}/4，危险阻断={blocked_count}/1")  # 输出教学集上的功能与安全指标


id | status | result_or_reason | visited_nodes
CODE-701 | allowed | 6440.0 | 3
CODE-702 | allowed | 2100.0 | 3
CODE-703 | allowed | 1073.33 | 8
CODE-704 | allowed | [980.0, 1350.0, 2100.0] | 7
CODE-705 | blocked | call_denied | 0
汇总：合法完成=4/4，危险阻断=1/1


## 结果解读

Capability VM 完成了总额、最大值、平均值和 Top-3 四个分析目标，并输出了平均值表达式的 AST 访问轨迹。导入加属性链在任何真实系统调用前返回 call_denied，长算术表达式被节点预算拒绝。安全的核心不是列一个危险词黑名单，而是只解释明确允许的语法和能力。

## 生产边界

生产代码沙箱仍需进程或容器隔离、只读文件系统、无网络默认策略、CPU/内存/墙钟限制、系统调用过滤、输出截断和审计。AST 白名单只适合极小表达式语言，不能安全承载完整 Python。真实数据还要按列级权限脱敏，并避免错误消息泄露环境信息。

## 最小回归测试

In [6]:
assert len(requests) >= 5  # 保证评测同时包含多个合法请求和越权请求
assert baseline_result == sum(expenses)  # 保证安全能力仍能完成基础费用求和
assert average_result == round(sum(expenses) / len(expenses), 2)  # 保证组合算术在白名单解释器中正确
assert malicious_error == "call_denied"  # 保证导入属性链在宿主调用前被拒绝
assert budget_error.startswith("budget_exceeded")  # 保证复杂表达式受静态节点预算约束
assert allowed_count == 4 and blocked_count == 1  # 保证四个合法目标完成且危险请求被阻断
